# PayPack: AI Agent Autonomous Payments with LangChain

This notebook demonstrates how to use **PayPack** to enable your LangChain agent to make autonomous on-chain payments via x402/AP2 protocols.

## What you'll learn
- Initialize PayPack with a signer
- Make ETH and USDC payments
- Auto-handle HTTP 402 responses
- Batch micro-payments with ERC-4337
- Use PayPack as a LangChain Tool

In [ ]:
# Install PayPack
!pip install langchain-paypack

## 1. Initialize AgentPay

In [ ]:
from paypack import AgentPay, LocalSigner

# Create a signer (use env var PRIVATE_KEY in production)
signer = LocalSigner(private_key="0xYOUR_PRIVATE_KEY")

# Initialize payment client
pay = AgentPay(
    signer=signer,
    network="base-sepolia",        # Testnet - use "base-mainnet" for real payments
    spend_limit_daily=10.0,         # Max $10/day safety limit
    broadcast=False,                # Set True for real transactions
    max_retries=3,                  # Retry failed transactions
    limit_store=None,               # Uses InMemoryStore by default
)

print(f"Wallet: {pay.address}")
print(f"Network: {pay.network_name} (chain_id={pay.chain_id})")

## 2. Send ETH Payment

In [ ]:
receipt = pay.send(
    to="0x9cbF3Ca5185Ca55C804c2c4b726De212A17734F8",
    amount=0.0001,
    currency="ETH"
)
print(f"TX Hash: {receipt['tx_hash']}")
print(f"Status: {receipt['status']}")
print(f"Remaining daily limit: {receipt['daily_remaining']} ETH")

## 3. Send USDC Payment

In [ ]:
receipt = pay.send(
    to="0x9cbF3Ca5185Ca55C804c2c4b726De212A17734F8",
    amount=0.001,
    currency="USDC"
)
print(f"TX Hash: {receipt['tx_hash']}")
print(f"Status: {receipt['status']}")

## 4. Auto-Handle HTTP 402 (x402 Protocol)

When your agent hits a paid API endpoint, PayPack auto-detects the payment protocol and pays.

In [ ]:
import requests

# Simulate: your agent calls a premium API
response = requests.get("https://api.data-provider.com/premium-endpoint")

if response.status_code == 402:
    # PayPack handles x402 (Coinbase) and AP2 (Google) headers automatically
    content = pay.auto_handle_402(response)
    print(f"Paid and got content: {content}")

## 5. Batch Micro-Payments (ERC-4337)

Bundle multiple small payments into a single transaction to save gas fees.

In [ ]:
result = pay.batch_pay([
    {"to": "0xAddr1", "amount": 0.001, "currency": "USDC"},
    {"to": "0xAddr2", "amount": 0.002, "currency": "USDC"},
    {"to": "0xAddr3", "amount": 0.001, "currency": "ETH"},
])
print(f"Batch size: {result['batch_size']}")
print(f"Total amount: {result['total_amount']}")
print(f"Status: {result['status']}")

## 6. PayPack as a LangChain Tool

Integrate PayPack directly into your LangChain agent as a callable tool.

In [ ]:
from langchain_paypack import PayPackTool

tool = PayPackTool(
    private_key="0xYOUR_PRIVATE_KEY",
    wallet_address="0xYOUR_ADDRESS",
    network="base-sepolia",
    spend_limit_daily=10.0,
    broadcast=False
)

# Your LangChain agent can now call:
result = tool._run(
    to="0xRecipient",
    amount=0.001,
    currency="USDC"
)
print(result)

## 7. Production Setup (AWS KMS)

For production, never expose private keys. Use AWS KMS:

In [ ]:
from paypack import AgentPay, AWSKMSSigner

# Private key NEVER leaves AWS HSM
signer = AWSKMSSigner(
    key_id="alias/paypack-eth-key",
    region="us-east-1"
)

pay = AgentPay(
    signer=signer,
    network="base-mainnet",
    spend_limit_daily=1000.0,
    broadcast=True,
    limit_store="redis",  # Persistent limits
    max_retries=5,
)

print(f"Production wallet: {pay.address}")

## Summary

- ✅ Single payments: `pay.send()`
- ✅ HTTP 402 auto-handling: `pay.auto_handle_402()`
- ✅ Batch settlement: `pay.batch_pay()`
- ✅ LangChain Tool: `PayPackTool`
- ✅ Production KMS: `AWSKMSSigner`
- ✅ RPC failover, retry, persistent limits

**GitHub**: https://github.com/rhcjw/paypack  
**PyPI**: https://pypi.org/project/langchain-paypack/